## **05 - ETL: S3 vers AWS RDS (PostgreSQL)**

Objectif: charger les CSV propres stockes sur S3 dans une base PostgreSQL AWS RDS pour faciliter les analyses.

**Flux ETL**:
1. Extract: lecture des CSV depuis S3
2. Transform: nettoyage leger des noms de colonnes/tables
3. Load: insertion dans PostgreSQL RDS (tables remplacees)

In [10]:
import os
import re
from io import StringIO
from pathlib import Path
from datetime import datetime, timezone

import boto3
import pandas as pd
from dotenv import load_dotenv
from botocore.exceptions import ClientError
from sqlalchemy import create_engine, text

In [11]:
# Chargement des variables d'environnement
load_dotenv()

AWS_BUCKET_NAME = os.getenv("AWS_BUCKET_NAME")
AWS_REGION_NAME = os.getenv("AWS_REGION_NAME", "eu-west-3")
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

AWS_RDS_HOST = os.getenv("AWS_RDS_HOST")
AWS_RDS_PORT = os.getenv("AWS_RDS_PORT", "5432")
AWS_RDS_USER = os.getenv("AWS_RDS_USER")
AWS_RDS_PASSWORD = os.getenv("AWS_RDS_PASSWORD")
AWS_RDS_DATABASE = os.getenv("AWS_RDS_DATABASE")

required_vars = {
    "AWS_BUCKET_NAME": AWS_BUCKET_NAME,
    "AWS_ACCESS_KEY": AWS_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
    "AWS_RDS_HOST": AWS_RDS_HOST,
    "AWS_RDS_USER": AWS_RDS_USER,
    "AWS_RDS_PASSWORD": AWS_RDS_PASSWORD,
    "AWS_RDS_DATABASE": AWS_RDS_DATABASE,
}

missing = [k for k, v in required_vars.items() if not v]
if missing:
    raise ValueError(f"Variables manquantes dans .env: {missing}")

print("Configuration AWS/S3/RDS chargee.")
print(f"Bucket: {AWS_BUCKET_NAME}")
print(f"Region: {AWS_REGION_NAME}")
print(f"RDS host: {AWS_RDS_HOST}:{AWS_RDS_PORT}")

Configuration AWS/S3/RDS chargee.
Bucket: amzn-jedha-39
Region: eu-west-3
RDS host: kayak-db.c7ykk4ckcpd2.eu-west-3.rds.amazonaws.com:5432


In [12]:
# Configuration ETL
S3_PREFIX = "datasets/kayak"
TARGET_SCHEMA = "public"

TABLE_NAME_OVERRIDES = {
    "top20_booking_hotels_enriched.csv": "top20_booking_hotels_enriched",
    "cities_weather_7d_raw.csv": "cities_weather_7d_raw",
    "cities_weather_scored.csv": "cities_weather_scored",
    "cities_geocoded.csv": "cities_geocoded",
}

def normalize_table_name(file_name: str) -> str:
    base = Path(file_name).stem.lower()
    base = re.sub(r"[^a-z0-9_]+", "_", base)
    base = re.sub(r"_+", "_", base).strip("_")
    if not base:
        raise ValueError(f"Nom de table invalide pour {file_name}")
    if base[0].isdigit():
        base = f"t_{base}"
    return base

In [13]:
# Client S3 + inventaire des CSV
s3_client = boto3.client(
    "s3",
    region_name=AWS_REGION_NAME,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

try:
    s3_client.head_bucket(Bucket=AWS_BUCKET_NAME)
except ClientError as e:
    raise RuntimeError(f"Impossible d'acceder au bucket '{AWS_BUCKET_NAME}': {e}")

resp = s3_client.list_objects_v2(Bucket=AWS_BUCKET_NAME, Prefix=f"{S3_PREFIX}/")
csv_keys = sorted([obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".csv")])

if not csv_keys:
    raise FileNotFoundError(f"Aucun CSV trouve dans s3://{AWS_BUCKET_NAME}/{S3_PREFIX}/")

print("CSV detectes sur S3:")
for k in csv_keys:
    print(f"- s3://{AWS_BUCKET_NAME}/{k}")

csv_keys

CSV detectes sur S3:
- s3://amzn-jedha-39/datasets/kayak/cities_geocoded.csv
- s3://amzn-jedha-39/datasets/kayak/cities_weather_7d_raw.csv
- s3://amzn-jedha-39/datasets/kayak/cities_weather_scored.csv
- s3://amzn-jedha-39/datasets/kayak/top20_booking_hotels_enriched.csv


['datasets/kayak/cities_geocoded.csv',
 'datasets/kayak/cities_weather_7d_raw.csv',
 'datasets/kayak/cities_weather_scored.csv',
 'datasets/kayak/top20_booking_hotels_enriched.csv']

In [14]:
# Extract: lecture des CSV S3 en DataFrames
dataframes = {}
extract_log = []

for key in csv_keys:
    obj = s3_client.get_object(Bucket=AWS_BUCKET_NAME, Key=key)
    content = obj["Body"].read().decode("utf-8")
    df = pd.read_csv(StringIO(content))

    # Normalisation des noms de colonnes pour SQL
    df.columns = [re.sub(r"[^a-zA-Z0-9_]+", "_", c.strip().lower()) for c in df.columns]

    file_name = Path(key).name
    table_name = TABLE_NAME_OVERRIDES.get(file_name, normalize_table_name(file_name))

    dataframes[table_name] = df
    extract_log.append({
        "s3_key": key,
        "table_name": table_name,
        "row_count": len(df),
        "column_count": len(df.columns),
    })

extract_df = pd.DataFrame(extract_log)
extract_df

,s3_key,table_name,row_count,column_count
0,datasets/kayak/cities_geocoded.csv,cities_geocoded,35,11
1,datasets/kayak/cities_weather_7d_raw.csv,cities_weather_7d_raw,35,10
2,datasets/kayak/cities_weather_scored.csv,cities_weather_scored,35,17
3,datasets/kayak/top20_booking_hotels_enriched.csv,top20_booking_hotels_enriched,20,8


In [15]:
# Connexion PostgreSQL RDS
db_url = (
    f"postgresql+psycopg2://{AWS_RDS_USER}:{AWS_RDS_PASSWORD}"
    f"@{AWS_RDS_HOST}:{AWS_RDS_PORT}/{AWS_RDS_DATABASE}"
)

engine = create_engine(db_url, pool_pre_ping=True)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Connexion RDS OK.")

Connexion RDS OK.


In [16]:
# Load: insertion DataFrame -> PostgreSQL
load_log = []

for table_name, df in dataframes.items():
    # if_exists='replace': supprime/recree la table pour un chargement idempotent
    df.to_sql(
        name=table_name,
        con=engine,
        schema=TARGET_SCHEMA,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=1000,
    )

    with engine.connect() as conn:
        inserted_rows = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_SCHEMA}.{table_name}")).scalar()

    load_log.append({
        "table_name": table_name,
        "rows_loaded": int(inserted_rows),
        "loaded_at_utc": datetime.now(timezone.utc).isoformat(),
    })

load_df = pd.DataFrame(load_log)
load_df

,table_name,rows_loaded,loaded_at_utc
0,cities_geocoded,35,2026-04-23T03:28:54.481195+00:00
1,cities_weather_7d_raw,35,2026-04-23T03:29:00.783305+00:00
2,cities_weather_scored,35,2026-04-23T03:29:07.290397+00:00
3,top20_booking_hotels_enriched,20,2026-04-23T03:29:13.890489+00:00


In [17]:
# Verification rapide des tables chargees
with engine.connect() as conn:
    table_counts = []
    for table_name in load_df["table_name"]:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_SCHEMA}.{table_name}")).scalar()
        table_counts.append({"table_name": table_name, "row_count": int(n)})

verify_df = pd.DataFrame(table_counts).sort_values("table_name")
verify_df

,table_name,row_count
0,cities_geocoded,35
1,cities_weather_7d_raw,35
2,cities_weather_scored,35
3,top20_booking_hotels_enriched,20


In [18]:
# Export des journaux ETL
logs_dir = Path("../outputs/logs")
logs_dir.mkdir(parents=True, exist_ok=True)

extract_log_file = logs_dir / "etl_extract_s3.csv"
load_log_file = logs_dir / "etl_load_rds.csv"

extract_df.to_csv(extract_log_file, index=False)
load_df.to_csv(load_log_file, index=False)

print(f"Extract log: {extract_log_file.resolve()}")
print(f"Load log: {load_log_file.resolve()}")

Extract log: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/logs/etl_extract_s3.csv
Load log: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/logs/etl_load_rds.csv
